# 04 - Interpretability and Business Insights

This notebook translates model evidence into customer prioritization, campaign planning, and executive-facing recommendations.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

## Feature Drivers

In [2]:
import pandas as pd

importance = pd.read_csv(PROJECT_ROOT / "outputs" / "reports" / "feature_importance.csv")
importance.head(15)

,feature,importance_mean,importance_std
0,Previously_Insured,0.053241,0.003586
1,Policy_Sales_Channel_response_rate_te,0.048324,0.003320
2,customer_risk_segment,0.040406,0.001733
3,vehicle_damage_flag,0.039132,0.002464
4,Age,0.032740,0.003744
5,cross_sell_opportunity_segment,0.028860,0.004010
6,vehicle_age_encoded,0.017374,0.002831
7,Vehicle_Age,0.012983,0.001205
8,Policy_Sales_Channel,0.008765,0.002065
9,Region_Code_response_rate_te,0.004042,0.002675


## Ranked Customer Scores

In [3]:
scores = pd.read_csv(PROJECT_ROOT / "outputs" / "predictions" / "test_customer_scores.csv")
scores[
    [
        "id",
        "propensity_score",
        "propensity_decile",
        "age_group",
        "premium_band",
        "cross_sell_opportunity_segment",
        "targeting_recommendation",
    ]
].sort_values("propensity_score", ascending=False).head(10)

,id,propensity_score,propensity_decile,age_group,premium_band,cross_sell_opportunity_segment,targeting_recommendation
101911,483021,0.926129,1,46-55,Premium,Priority,Highest priority campaign
120827,501937,0.918596,1,46-55,Upper Core,Priority,Highest priority campaign
91563,472673,0.918596,1,46-55,Upper Core,Priority,Highest priority campaign
72692,453802,0.918151,1,46-55,High,Priority,Highest priority campaign
85587,466697,0.918044,1,46-55,Upper Core,Priority,Highest priority campaign
124191,505301,0.915539,1,46-55,High,Priority,Highest priority campaign
44678,425788,0.914956,1,36-45,Upper Core,Priority,Highest priority campaign
51511,432621,0.914884,1,36-45,Upper Core,Priority,Highest priority campaign
60903,442013,0.914156,1,36-45,Upper Core,Priority,Highest priority campaign
93504,474614,0.913577,1,36-45,Upper Core,Priority,Highest priority campaign


## Campaign Priority Mix

In [4]:
(
    scores.groupby(["propensity_decile", "targeting_recommendation"], observed=True)
    .agg(customers=("id", "size"), average_score=("propensity_score", "mean"))
    .reset_index()
    .sort_values(["propensity_decile", "customers"])
)

,propensity_decile,targeting_recommendation,customers,average_score
0,1,Highest priority campaign,12704,0.816622
1,2,Scaled priority outreach,12704,0.765713
2,3,Scaled priority outreach,12703,0.708584
3,4,Nurture / test campaign,12704,0.589165
4,5,Nurture / test campaign,12704,0.352039
5,6,Low priority holdout,12703,0.067851
6,7,Low priority holdout,12704,0.003361
7,8,Low priority holdout,12703,0.001951
8,9,Low priority holdout,12704,0.001456
9,10,Low priority holdout,12704,0.001075


## Executive Interpretation

In [5]:
interpretation_report = PROJECT_ROOT / "outputs" / "reports" / "interpretability_report.md"
print(interpretation_report.read_text(encoding="utf-8")[:2_000])

# Model Interpretability Report

Primary interpretability artifact: **permutation importance**.

Permutation importance is used for the committed report because it is lightweight, deterministic, and works with the deployed scikit-learn pipeline. SHAP remains available as an optional local extension for deeper individual-level analysis.

## Global Drivers
- `Previously_Insured`: 0.05324
- `Policy_Sales_Channel_response_rate_te`: 0.04832
- `customer_risk_segment`: 0.04041
- `vehicle_damage_flag`: 0.03913
- `Age`: 0.03274
- `cross_sell_opportunity_segment`: 0.02886
- `vehicle_age_encoded`: 0.01737
- `Vehicle_Age`: 0.01298
- `Policy_Sales_Channel`: 0.00877
- `Region_Code_response_rate_te`: 0.00404

## Business Interpretation
- Previous vehicle damage and not-yet-insured status are expected to be strong positive signals for cross-sell relevance.
- Vehicle age, customer age band, annual premium band, region, and sales channel help prioritize customer groups, but they should be treated as ass